In [2]:
import numpy as np

words = ["my", "student", "loan", "payment", "late"]
scores = np.array([0.1, 2.5, 2.7, 1.0, 0.2])

def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()

weights = softmax(scores)

for w, s, a in zip(words, scores, weights):
    print(f"{w:>8}  score {s:4.1f}  weight {a:.3f}")
print("sum of weights:", weights.sum())

      my  score  0.1  weight 0.034
 student  score  2.5  weight 0.379
    loan  score  2.7  weight 0.463
 payment  score  1.0  weight 0.085
    late  score  0.2  weight 0.038
sum of weights: 1.0


In [3]:
vectors = np.array([
    [0.0, 0.0, 1.0],
    [1.0, 0.2, 0.0],
    [0.9, 0.3, 0.1],
    [0.3, 1.0, 0.0],
    [0.0, 0.8, 0.2],
])
dims = ["loan", "payment", "generic"]

weighted = weights @ vectors
plain = vectors.mean(axis=0)

print(f"{'':>9}" + "".join(f"{d:>9}" for d in dims))
print(f"{'weighted':>9}" + "".join(f"{v:9.3f}" for v in weighted))
print(f"{'plain':>9}" + "".join(f"{v:9.3f}" for v in plain))

              loan  payment  generic
 weighted    0.822    0.330    0.088
    plain    0.440    0.460    0.260


In [4]:
scores_2 = scores.copy()
scores_2[3] = 2.0
weights_2 = softmax(scores_2)

for w, a1, a2 in zip(words, weights, weights_2):
    print(f"{w:>8}  before {a1:.3f}  after {a2:.3f}")

weighted_2 = weights_2 @ vectors
print(f"{'':>9}" + "".join(f"{d:>9}" for d in dims))
print(f"{'before':>9}" + "".join(f"{v:9.3f}" for v in weighted))
print(f"{'after':>9}" + "".join(f"{v:9.3f}" for v in weighted_2))

      my  before 0.034  after 0.030
 student  before 0.379  after 0.331
    loan  before 0.463  after 0.405
 payment  before 0.085  after 0.201
    late  before 0.038  after 0.033
              loan  payment  generic
   before    0.822    0.330    0.088
    after    0.756    0.415    0.077


In [5]:
n_filler = 20
scores_long = np.concatenate([scores, np.full(n_filler, -2.0)])
vectors_long = np.vstack([vectors, np.tile([0.0, 0.0, 1.0], (n_filler, 1))])

weights_long = softmax(scores_long)

raw_short = vectors.sum(axis=0)
raw_long = vectors_long.sum(axis=0)
weighted_long = weights_long @ vectors_long

print(f"{'':>15}" + "".join(f"{d:>9}" for d in dims))
for name, v in [("raw short", raw_short), ("raw long", raw_long),
                ("weighted short", weighted), ("weighted long", weighted_long)]:
    print(f"{name:>15}" + "".join(f"{x:9.3f}" for x in v))
print("loan weight short vs long:", round(weights[2], 3), round(weights_long[2], 3))

                    loan  payment  generic
      raw short    2.200    2.300    1.300
       raw long    2.200    2.300   21.300
 weighted short    0.822    0.330    0.088
  weighted long    0.758    0.304    0.159
loan weight short vs long: 0.463 0.427


In [6]:
import keras
import numpy as np

keras.utils.set_random_seed(0)
probe = keras.Sequential([
    keras.Input(shape=(None,), dtype="int32"),
    keras.layers.Embedding(10, 3, mask_zero=True),
    keras.layers.GlobalAveragePooling1D(),
])

short = probe(np.array([[4, 7]])).numpy()
padded = probe(np.array([[4, 7, 0, 0, 0, 0, 0, 0]])).numpy()
print("short :", short.round(4))
print("padded:", padded.round(4))
print("equal :", np.allclose(short, padded))

I0000 00:00:1791013008.733332   13913 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791013009.254145   13913 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791013012.033615   13913 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791013015.888255   13913 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3536 MB mem

short : [[0.0104 0.008  0.0008]]
padded: [[0.0104 0.008  0.0008]]
equal : True


In [7]:
import sys
from pathlib import Path
import keras
import numpy as np

root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(root / "src"))
import tensorflow as tf
from model import split_with_bigrams

print(split_with_bigrams(tf.constant(["my student loan is late"])).to_list())
from model import AttentionPooling

print("src path:", root / "src")
keras.utils.set_random_seed(0)
probe = keras.Sequential([
    keras.Input(shape=(None,), dtype="int32"),
    keras.layers.Embedding(10, 3, mask_zero=True),
    AttentionPooling(),
])

short = probe(np.array([[4, 7]])).numpy()
padded = probe(np.array([[4, 7, 0, 0, 0, 0, 0, 0]])).numpy()
print("short :", short.round(4))
print("padded:", padded.round(4))
print("equal :", np.allclose(short, padded))

[[b'my', b'student', b'loan', b'is', b'late', b'my_student', b'student_loan', b'loan_is', b'is_late']]
src path: /mnt/c/Users/Baraa Hereh/Desktop/machine/complaint-routing/src
short : [[ 0.0101  0.0077 -0.0003]]
padded: [[ 0.0101  0.0077 -0.0003]]
equal : True


In [8]:
import tensorflow as tf
from model import split_with_bigrams

print(split_with_bigrams(tf.constant(["my student loan is late"])).to_list())

[[b'my', b'student', b'loan', b'is', b'late', b'my_student', b'student_loan', b'loan_is', b'is_late']]


In [9]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(root / "src"))
from baseline import load_config, load_split, run_name

cfg = load_config(root / "configs/config.yaml")
d, c, b = cfg["data"], cfg["clean"], cfg["baseline"]
model = joblib.load(root / b["model_dir"] / f"{run_name(b)}.joblib")
X_va, y_va = load_split(root / c["out_dir"], "val", d["text_col"], c["label_col"])
pred = model.predict(X_va)

labels = list(model.classes_)
cm = pd.DataFrame(confusion_matrix(y_va, pred, labels=labels), index=labels, columns=labels)
pairs = (cm.where(~np.eye(len(labels), dtype=bool)).stack()
           .sort_values(ascending=False).head(6))
print(pairs.to_string())

Credit reporting, credit repair services, or other personal consumer reports  Debt collection                                                                 1019.0
Debt collection                                                               Credit reporting, credit repair services, or other personal consumer reports     615.0
Credit reporting, credit repair services, or other personal consumer reports  Credit card or prepaid card                                                      338.0
                                                                              Mortgage                                                                         190.0
Credit card or prepaid card                                                   Credit reporting, credit repair services, or other personal consumer reports     189.0
Credit reporting, credit repair services, or other personal consumer reports  Vehicle loan or lease                                                            184.0


In [10]:
mask = (y_va.to_numpy() == labels[2]) & (pred == labels[3])
sample = X_va[mask].sample(10, random_state=0)
for i, text in enumerate(sample, 1):
    print(f"--- {i} ---")
    print(text[:500])
    print()

--- 1 ---
Settled/paid a charge off with an collection company for comenity cb/ my place rewards and comenity refuses to update the status of the account along with the comment. Status of the account should state ; current. Comments should state ; paid as agreed or paid in full for less than full balance. Ive tried to dispute the account multiple times and Ive tried to contact, but Comenity refuses to talk to me. Comenity states that they dont own my account, but refuses to remove it from my report or upd

--- 2 ---
account I am not responsible for being reported on my [PII] credit report by plaza services. Not mine and claimed original creditor [PII] can not even legally lend online in virginia.

--- 3 ---
As I have not heard back from TransUnion in over 60 days, regarding my demand for proof, since my notice of dispute dated [PII]/[PII]/2015 and you have not supplied the demanded proof of the alleged debt, under the doctrine of estoppel by silence, Engelhardt v Gravens ( Mo ) 281 SW 

In [11]:
proba = model.predict_proba(X_va)
conf = proba.max(axis=1)
correct = model.classes_[proba.argmax(axis=1)] == y_va.to_numpy()

bins = pd.cut(conf, [0, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
calib = pd.DataFrame({"conf": conf, "correct": correct}).groupby(bins, observed=True).agg(
    n=("conf", "size"), mean_conf=("conf", "mean"), accuracy=("correct", "mean"))
print(calib.round(3).to_string())
print()

for t in [0.5, 0.6, 0.7, 0.8, 0.9]:
    auto = conf >= t
    print(f"threshold {t:.1f} | auto-routed {auto.mean():6.1%} | accuracy on those {correct[auto].mean():6.1%}")

                n  mean_conf  accuracy
(0.0, 0.4]   1535      0.322     0.441
(0.4, 0.5]   1839      0.454     0.594
(0.5, 0.6]   2295      0.551     0.657
(0.6, 0.7]   2598      0.651     0.728
(0.7, 0.8]   3045      0.751     0.812
(0.8, 0.9]   4517      0.854     0.880
(0.9, 1.0]  10680      0.961     0.957

threshold 0.5 | auto-routed  87.3% | accuracy on those  86.8%
threshold 0.6 | auto-routed  78.6% | accuracy on those  89.1%
threshold 0.7 | auto-routed  68.8% | accuracy on those  91.4%
threshold 0.8 | auto-routed  57.3% | accuracy on those  93.4%
threshold 0.9 | auto-routed  40.3% | accuracy on those  95.7%


In [12]:
import time
from predict import load_model, route

model, names = load_model(root / "configs/config.yaml")
texts = X_va.sample(500, random_state=0).tolist()
for t in texts[:20]:
    route(model, names, t)

times = []
for t in texts:
    start = time.perf_counter()
    route(model, names, t)
    times.append((time.perf_counter() - start) * 1000)
print(f"median {np.median(times):.2f} ms | p95 {np.percentile(times, 95):.2f} ms | max {max(times):.2f} ms")

median 1.24 ms | p95 2.05 ms | max 5.79 ms
